# Convergence-plot reproduction

This notebook reproduces the manuscript convergence plots from 10 independent runs of Line Search, BO-DSO, and SAASBO-DSO. Solid curves show the mean best-so-far transition probability and shaded regions show ±1 sample standard deviation across runs. Input logs are read from `data/convergence_logs/` and generated figures are written to `figures/convergence/`.


In [ ]:
from pathlib import Path
import ast, re
import numpy as np
import matplotlib as mpl
mpl.use("pgf")
import matplotlib.pyplot as plt
plt.ioff()
mpl.rcParams.update({"pgf.texsystem":"pdflatex","text.usetex":True,"font.family":"serif","pgf.rcfonts":False})

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "convergence_logs").is_dir():
            return candidate
    raise FileNotFoundError("Could not find data/convergence_logs inside the repository.")

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data" / "convergence_logs"
PLOT_DIR = REPO_ROOT / "figures" / "convergence"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print("Input data:", DATA_DIR)
print("Output figures:", PLOT_DIR)


In [ ]:
def search_string_in_file(file_name, string_to_search):
    """Search for the given string in file and return lines containing that string,
    along with line numbers"""
    line_number = 0
    list_of_results = []
    # Open the file in read only mode
    with open(file_name, 'r') as read_obj:
        # Read all lines in the file one by one
        for line in read_obj:
            # For each line, check if line contains the string
            line_number += 1
            if string_to_search in line:
                # If yes, then add the line number & line as a tuple in the list
                list_of_results.append((line_number, line.rstrip()))
    # Return list of tuples containing line numbers and lines where string is found
    return list_of_results

In [ ]:
def get_array(filename,str):
  matched_lines = search_string_in_file(filename, str)
  #print('Total Matched lines : ', len(matched_lines))
  for elem in matched_lines:
    #print('Line Number = ', elem[0], ' :: Line = ', elem[1])
    lines = elem[1]
  sub_list =["best found:","best Ps:","y-vals:","eval time:","total time:","Best P:","Eval Time:"]
  for sub in sub_list:
    lines = lines.replace(sub, ' ')
  res = " ".join(lines.split())
  res = ast.literal_eval(res) 
  return np.array(res)

In [ ]:
# Verify that all logs required for the convergence figures are present.
TRANSITIONS = ["hex_1to2","hex_4to6","triG_1to2","triG_2to3","triG_3to4","triN_1to2","triN_1to3","triN_2to4","triN_4to6","triN_9to10"]
required_files=[]
for prefix in TRANSITIONS:
    for i in range(1,11):
        required_files += [
            DATA_DIR / f"{prefix}_linesearchtimedomain_200iter_{i}.txt",
            DATA_DIR / f"{prefix}_bodaw_100iter_ittexp10_mloop1000_gradTrue_{i}.txt",
            DATA_DIR / f"{prefix}_saasbononaxisaligned_100iter_ittexp10_{i}.txt",
        ]
missing=[p for p in required_files if not p.exists()]
print(f"Required logs: {len(required_files)}; missing: {len(missing)}")
if missing:
    raise FileNotFoundError("Missing required convergence logs; first missing: " + str(missing[0]))


In [ ]:
def extract_metrics(prefix, m):

    # Initialize lists
    linesearch_list_Y = []
    linesearch_list_T = []
    bodaw_list_Y = []
    bodaw_list_T = []
    #saasbo_list_Y = []          # plain SAASBO
    #saasbo_list_T = []
    #saasboexpand_list_Y = []    # SAASBOEXPAND
    #saasboexpand_list_T = []
    snonaasboexpand_list_Y = []
    snonaasboexpand_list_T = []

    for i in range(m):
        # ---- LineSearch ----
        linesearch_file = DATA_DIR / f"{prefix}_linesearchtimedomain_200iter_{i+1}.txt"
        arraylY = get_array(linesearch_file, "Best P:")
        arraylT = get_array(linesearch_file, "Eval Time:")
        linesearch_list_Y.append(arraylY)
        linesearch_list_T.append(arraylT)

        # ---- BODAW ----
        bodaw_file = DATA_DIR / f"{prefix}_bodaw_100iter_ittexp10_mloop1000_gradTrue_{i+1}.txt"
        arrayboY = get_array(bodaw_file, "best Ps:")
        arrayboT = get_array(bodaw_file, "eval time:")
        bodaw_list_Y.append(arrayboY)
        bodaw_list_T.append(arrayboT)

        # ---- SNONAASBOEXPAND ----
        snonaasboexpand_file = DATA_DIR / f"{prefix}_saasbononaxisaligned_100iter_ittexp10_{i+1}.txt"
        arraysnaY = get_array(snonaasboexpand_file, "best Ps:")
        arraysnaT = get_array(snonaasboexpand_file, "eval time:")
        snonaasboexpand_list_Y.append(arraysnaY)
        snonaasboexpand_list_T.append(arraysnaT)

    # Helper: truncate & average one method's curves separately
    def truncate_and_avg(list_Y, list_T):
        min_len = min(
            min(len(a) for a in list_Y),
            min(len(a) for a in list_T)
        )
    
        Y_runs = np.array([
            a[:min_len] for a in list_Y
        ])
    
        T_runs = np.array([
            a[:min_len] for a in list_T
        ])
    
        bestY = np.mean(Y_runs, axis=0)
        T = np.mean(T_runs, axis=0)
    
        # Sample standard deviation across the independent runs
        SD = np.std(
            Y_runs,
            axis=0,
            ddof=1
        )

        return bestY, T, SD, min_len

    # Compute per-method truncated averages + lengths
    linesearch_bestY, linesearch_T, linesearch_SD, len_ls = truncate_and_avg(
    linesearch_list_Y, linesearch_list_T
)

    bodaw_bestY, bodaw_T, bodaw_SD, len_bo = truncate_and_avg(
    bodaw_list_Y, bodaw_list_T
)

    snonaasboexpand_bestY, snonaasboexpand_T, snonaasboexpand_SD, len_snon = truncate_and_avg(
    snonaasboexpand_list_Y, snonaasboexpand_list_T
)
    
    print("Lengths after truncation:")
    print(f"  Line search       : {len_ls}")
    print(f"  BO expand         : {len_bo}")
    #print(f"  SAASBO            : {len_saas}")
    #print(f"  SAASBO expand     : {len_saasexp}")
    print(f"  SNonAASBO expand  : {len_snon}")
    

    return (
    linesearch_bestY, linesearch_T, linesearch_SD,
    bodaw_bestY, bodaw_T, bodaw_SD,
    snonaasboexpand_bestY, snonaasboexpand_T, snonaasboexpand_SD
)


In [ ]:
m = 10

# ===================== hex =====================

hex1to2linesearchbestY, hex1to2linesearchT, hex1to2linesearchSD, \
hex1to2bodawbestY, hex1to2bodawT, hex1to2bodawSD, \
hex1to2snonaasboexpandbestY, hex1to2snonaasboexpandT, hex1to2snonaasboexpandSD = \
    extract_metrics("hex_1to2", m)

hex4to6linesearchbestY, hex4to6linesearchT, hex4to6linesearchSD, \
hex4to6bodawbestY, hex4to6bodawT, hex4to6bodawSD, \
hex4to6snonaasboexpandbestY, hex4to6snonaasboexpandT, hex4to6snonaasboexpandSD = \
    extract_metrics("hex_4to6", m)


# ===================== triG =====================

triG1to2linesearchbestY, triG1to2linesearchT, triG1to2linesearchSD, \
triG1to2bodawbestY, triG1to2bodawT, triG1to2bodawSD, \
triG1to2snonaasboexpandbestY, triG1to2snonaasboexpandT, triG1to2snonaasboexpandSD = \
    extract_metrics("triG_1to2", m)

triG2to3linesearchbestY, triG2to3linesearchT, triG2to3linesearchSD, \
triG2to3bodawbestY, triG2to3bodawT, triG2to3bodawSD, \
triG2to3snonaasboexpandbestY, triG2to3snonaasboexpandT, triG2to3snonaasboexpandSD = \
    extract_metrics("triG_2to3", m)

triG3to4linesearchbestY, triG3to4linesearchT, triG3to4linesearchSD, \
triG3to4bodawbestY, triG3to4bodawT, triG3to4bodawSD, \
triG3to4snonaasboexpandbestY, triG3to4snonaasboexpandT, triG3to4snonaasboexpandSD = \
    extract_metrics("triG_3to4", m)


# ===================== triN =====================

triN1to2linesearchbestY, triN1to2linesearchT, triN1to2linesearchSD, \
triN1to2bodawbestY, triN1to2bodawT, triN1to2bodawSD, \
triN1to2snonaasboexpandbestY, triN1to2snonaasboexpandT, triN1to2snonaasboexpandSD = \
    extract_metrics("triN_1to2", m)

triN1to3linesearchbestY, triN1to3linesearchT, triN1to3linesearchSD, \
triN1to3bodawbestY, triN1to3bodawT, triN1to3bodawSD, \
triN1to3snonaasboexpandbestY, triN1to3snonaasboexpandT, triN1to3snonaasboexpandSD = \
    extract_metrics("triN_1to3", m)

triN2to4linesearchbestY, triN2to4linesearchT, triN2to4linesearchSD, \
triN2to4bodawbestY, triN2to4bodawT, triN2to4bodawSD, \
triN2to4snonaasboexpandbestY, triN2to4snonaasboexpandT, triN2to4snonaasboexpandSD = \
    extract_metrics("triN_2to4", m)

triN4to6linesearchbestY, triN4to6linesearchT, triN4to6linesearchSD, \
triN4to6bodawbestY, triN4to6bodawT, triN4to6bodawSD, \
triN4to6snonaasboexpandbestY, triN4to6snonaasboexpandT, triN4to6snonaasboexpandSD = \
    extract_metrics("triN_4to6", m)

triN9to10linesearchbestY, triN9to10linesearchT, triN9to10linesearchSD, \
triN9to10bodawbestY, triN9to10bodawT, triN9to10bodawSD, \
triN9to10snonaasboexpandbestY, triN9to10snonaasboexpandT, triN9to10snonaasboexpandSD = \
    extract_metrics("triN_9to10", m)

In [ ]:

MIN_LEN_LINESEARCH      = 100   # example value
MIN_LEN_BODAW       = 100   # example value
#MIN_LEN_SAASBO          = 203   
#MIN_LEN_SAASBOEXPAND    = 186   # example value
MIN_LEN_SNONAASBOEXPAND = 100   # example value


In [ ]:
import numpy as np

def truncate_to_time(T, Y, SD, t_end):
    if len(T) == 0:
        return T, Y, SD

    T = np.asarray(T)
    Y = np.asarray(Y)
    SD = np.asarray(SD)

    mask = T <= t_end

    return (
        T[mask],
        Y[mask],
        SD[mask]
    )

In [ ]:
hex_configs = [

    (
        "hex1to2",

        hex1to2linesearchT,
        hex1to2linesearchbestY,
        hex1to2linesearchSD,

        hex1to2bodawT,
        hex1to2bodawbestY,
        hex1to2bodawSD,

        hex1to2snonaasboexpandT,
        hex1to2snonaasboexpandbestY,
        hex1to2snonaasboexpandSD
    ),

    (
        "hex4to6",

        hex4to6linesearchT,
        hex4to6linesearchbestY,
        hex4to6linesearchSD,

        hex4to6bodawT,
        hex4to6bodawbestY,
        hex4to6bodawSD,

        hex4to6snonaasboexpandT,
        hex4to6snonaasboexpandbestY,
        hex4to6snonaasboexpandSD
    ),
]


triG_configs = [

    (
        "triG1to2",

        triG1to2linesearchT,
        triG1to2linesearchbestY,
        triG1to2linesearchSD,

        triG1to2bodawT,
        triG1to2bodawbestY,
        triG1to2bodawSD,

        triG1to2snonaasboexpandT,
        triG1to2snonaasboexpandbestY,
        triG1to2snonaasboexpandSD
    ),

    (
        "triG2to3",

        triG2to3linesearchT,
        triG2to3linesearchbestY,
        triG2to3linesearchSD,

        triG2to3bodawT,
        triG2to3bodawbestY,
        triG2to3bodawSD,

        triG2to3snonaasboexpandT,
        triG2to3snonaasboexpandbestY,
        triG2to3snonaasboexpandSD
    ),

    (
        "triG3to4",

        triG3to4linesearchT,
        triG3to4linesearchbestY,
        triG3to4linesearchSD,

        triG3to4bodawT,
        triG3to4bodawbestY,
        triG3to4bodawSD,

        triG3to4snonaasboexpandT,
        triG3to4snonaasboexpandbestY,
        triG3to4snonaasboexpandSD
    ),
]


triN_configs = [

    (
        "triN1to2",

        triN1to2linesearchT,
        triN1to2linesearchbestY,
        triN1to2linesearchSD,

        triN1to2bodawT,
        triN1to2bodawbestY,
        triN1to2bodawSD,

        triN1to2snonaasboexpandT,
        triN1to2snonaasboexpandbestY,
        triN1to2snonaasboexpandSD
    ),

    (
        "triN1to3",

        triN1to3linesearchT,
        triN1to3linesearchbestY,
        triN1to3linesearchSD,

        triN1to3bodawT,
        triN1to3bodawbestY,
        triN1to3bodawSD,

        triN1to3snonaasboexpandT,
        triN1to3snonaasboexpandbestY,
        triN1to3snonaasboexpandSD
    ),

    (
        "triN2to4",

        triN2to4linesearchT,
        triN2to4linesearchbestY,
        triN2to4linesearchSD,

        triN2to4bodawT,
        triN2to4bodawbestY,
        triN2to4bodawSD,

        triN2to4snonaasboexpandT,
        triN2to4snonaasboexpandbestY,
        triN2to4snonaasboexpandSD
    ),

    (
        "triN4to6",

        triN4to6linesearchT,
        triN4to6linesearchbestY,
        triN4to6linesearchSD,

        triN4to6bodawT,
        triN4to6bodawbestY,
        triN4to6bodawSD,

        triN4to6snonaasboexpandT,
        triN4to6snonaasboexpandbestY,
        triN4to6snonaasboexpandSD
    ),

    (
        "triN9to10",

        triN9to10linesearchT,
        triN9to10linesearchbestY,
        triN9to10linesearchSD,

        triN9to10bodawT,
        triN9to10bodawbestY,
        triN9to10bodawSD,

        triN9to10snonaasboexpandT,
        triN9to10snonaasboexpandbestY,
        triN9to10snonaasboexpandSD
    ),
]

In [ ]:
import os, re
import numpy as np
import matplotlib as mpl

# --- PGF setup (must be before importing pyplot in some environments)
mpl.use("pgf")
import matplotlib.pyplot as plt

# --- Paper-like rcParams (tweak if needed)
plt.rcParams.update({
    "pgf.texsystem": "pdflatex",
    "text.usetex": True,
    "pgf.rcfonts": False,

    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,

    "lines.linewidth": 1.5,
    "lines.markersize": 4.0,

    "savefig.bbox": "tight",
})

# --- Nice dark, paper-friendly colors
COLORS = {
    "Line Search": "#D55E00",   # vermillion
    "BO-DSO":      "#0072B2",   # deep blue
    "SAASBO-DSO":  "#009E73",   # bluish green
}

MARKERS = {
    "Line Search": "^",
    "BO-DSO":      "o",
    "SAASBO-DSO":  "s",
}

LINESTYLES = {
    "Line Search": "-",
    "BO-DSO":      "-",
    "SAASBO-DSO":  "-",
}


def plot_run(
    T_linesearch, Y_linesearch, SD_linesearch,
    T_bodaw, Y_bodaw, SD_bodaw,
    T_snonaasboexpand, Y_snonaasboexpand, SD_snonaasboexpand,
    title, label_prefix,
    min_lengths,     # (min_ls, min_bo, min_snon)
    ax=None,
    save_path=None,
    clamp_x=True,
):

    if ax is None:
        fig, ax = plt.subplots(figsize=(3.35, 2.4))

    # --- Unpack per-method global mins
    min_ls, min_bo, min_snon = min_lengths


    # ============================================================
    # Clamp by iteration count first
    # ============================================================

    q_ls = min(
        min_ls,
        len(T_linesearch),
        len(Y_linesearch),
        len(SD_linesearch)
    )

    q_bo = min(
        min_bo,
        len(T_bodaw),
        len(Y_bodaw),
        len(SD_bodaw)
    )

    q_snon = min(
        min_snon,
        len(T_snonaasboexpand),
        len(Y_snonaasboexpand),
        len(SD_snonaasboexpand)
    )


    # ============================================================
    # Convert to arrays
    # ============================================================

    T_ls = np.asarray(T_linesearch[:q_ls])
    Y_ls = np.asarray(Y_linesearch[:q_ls])
    SD_ls = np.asarray(SD_linesearch[:q_ls])

    T_bo = np.asarray(T_bodaw[:q_bo])
    Y_bo = np.asarray(Y_bodaw[:q_bo])
    SD_bo = np.asarray(SD_bodaw[:q_bo])

    T_snon = np.asarray(T_snonaasboexpand[:q_snon])
    Y_snon = np.asarray(Y_snonaasboexpand[:q_snon])
    SD_snon = np.asarray(SD_snonaasboexpand[:q_snon])


    # ============================================================
    # Clamp by common time horizon
    # ============================================================

    if clamp_x and len(T_ls) and len(T_bo) and len(T_snon):

        t_end = min(
            T_ls[-1],
            T_bo[-1],
            T_snon[-1]
        )

        T_ls, Y_ls, SD_ls = truncate_to_time(
            T_ls,
            Y_ls,
            SD_ls,
            t_end
        )

        T_bo, Y_bo, SD_bo = truncate_to_time(
            T_bo,
            Y_bo,
            SD_bo,
            t_end
        )

        T_snon, Y_snon, SD_snon = truncate_to_time(
            T_snon,
            Y_snon,
            SD_snon,
            t_end
        )

        ax.set_xlim(
            0.0,
            float(t_end)
        )

        ax.set_autoscalex_on(False)


    # ============================================================
    # Error bands: mean ± 1 sample SD
    # ============================================================

    ax.fill_between(
        T_ls,
        np.clip(Y_ls - SD_ls, 0.0, 1.0),
        np.clip(Y_ls + SD_ls, 0.0, 1.0),
        color=COLORS["Line Search"],
        alpha=0.40,
        linewidth=0
    )

    ax.fill_between(
        T_bo,
        np.clip(Y_bo - SD_bo, 0.0, 1.0),
        np.clip(Y_bo + SD_bo, 0.0, 1.0),
        color=COLORS["BO-DSO"],
        alpha=0.40,
        linewidth=0
    )

    ax.fill_between(
        T_snon,
        np.clip(Y_snon - SD_snon, 0.0, 1.0),
        np.clip(Y_snon + SD_snon, 0.0, 1.0),
        color=COLORS["SAASBO-DSO"],
        alpha=0.40,
        linewidth=0
    )


    # ============================================================
    # Mean curves — your original plotting style
    # ============================================================

    ax.plot(
        T_ls,
        Y_ls,
        color=COLORS["Line Search"],
        marker=MARKERS["Line Search"],
        linestyle=LINESTYLES["Line Search"],
        label="Line Search",
        markevery=max(1, len(T_ls)//12)
    )

    ax.plot(
        T_bo,
        Y_bo,
        color=COLORS["BO-DSO"],
        marker=MARKERS["BO-DSO"],
        linestyle=LINESTYLES["BO-DSO"],
        label="BO-DSO",
        markevery=max(1, len(T_bo)//12)
    )

    ax.plot(
        T_snon,
        Y_snon,
        color=COLORS["SAASBO-DSO"],
        marker=MARKERS["SAASBO-DSO"],
        linestyle=LINESTYLES["SAASBO-DSO"],
        label="SAASBO-DSO",
        markevery=max(1, len(T_snon)//12)
    )


    # ============================================================
    # Formatting
    # ============================================================

    ax.set_title(title)
    ax.set_xlabel("Simulation Time (sec)")
    ax.set_ylabel(r"$P^{\ast}$")

    ax.grid(True, alpha=0.3)

    ax.set_ylim(0.0, 1.0)
    ax.set_autoscaley_on(False)


    if save_path:
        ax.figure.savefig(save_path)


def save_legend_only(output_path, labels=("Line Search", "BO-DSO", "SAASBO-DSO"), ncol=3):
    """
    Save a separate legend-only PGF/PDF figure that you can place under plots in LaTeX.
    """
    fig = plt.figure(figsize=(6.8, 0.45))  # wide & short
    ax = fig.add_subplot(111)
    ax.axis("off")

    # Create dummy handles
    handles = []
    for lab in labels:
        h, = ax.plot(
            [], [],
            color=COLORS[lab],
            marker=MARKERS[lab],
            linestyle=LINESTYLES[lab],
            label=lab
        )
        handles.append(h)

    leg = fig.legend(
        handles=handles,
        labels=list(labels),
        loc="center",
        ncol=ncol,
        frameon=True,
        handlelength=2.0,
        columnspacing=1.5
    )

    fig.savefig(output_path, bbox_inches="tight")
    plt.close(fig)


def plot_group(group_label, configs):
    PLOT_DIR.mkdir(parents=True, exist_ok=True)

    # Per-method min-length tuple from your constants
    min_lengths = (
        MIN_LEN_LINESDARCH,
        MIN_LEN_BODAW,
        MIN_LEN_SNONAASBOEXPAND,
    )

    cols = 3
    rows = (len(configs) + cols - 1) // cols

    # --- Full-width, paper-friendly figure sizing:
    # 6.8 inches is typical two-column textwidth
    fig_width = 6.8
    fig_height_per_row = 2.1
    fig, axs = plt.subplots(rows, cols, figsize=(fig_width, rows * fig_height_per_row))
    axs = np.array(axs).reshape(-1)

    for idx, config in enumerate(configs):
        name = config[0]

        # Extract only numbers and build "1 → 2"
        nums = re.findall(r'\d+', name)
        pretty_name = rf"{nums[0]} $\rightarrow$ {nums[1]}"
        title = f"{pretty_name} transition"

        # Unpack config: (name, T_ls, Y_ls, T_bodaw, Y_bodaw, T_snon, Y_snon)
        #_, T_ls, Y_ls, T_bodaw, Y_bodaw, T_snon, Y_snon = config
        
        # Unpack config:
        # (name,
        #  T_ls, Y_ls, SD_ls,
        #  T_bodaw, Y_bodaw, SD_bodaw,
        #  T_snon, Y_snon, SD_snon)
        
        (
            _,
            T_ls, Y_ls, SD_ls,
            T_bodaw, Y_bodaw, SD_bodaw,
            T_snon, Y_snon, SD_snon
        ) = config

        plot_run(
        T_linesearch=T_ls,
        Y_linesearch=Y_ls,
        SD_linesearch=SD_ls,
    
        T_bodaw=T_bodaw,
        Y_bodaw=Y_bodaw,
        SD_bodaw=SD_bodaw,
    
        T_snonaasboexpand=T_snon,
        Y_snonaasboexpand=Y_snon,
        SD_snonaasboexpand=SD_snon,
    
        title=title,
        label_prefix=pretty_name,
        min_lengths=min_lengths,
        ax=axs[idx],
        save_path=None,
        clamp_x=True,
    )

    # Delete unused axes
    for j in range(len(configs), len(axs)):
        fig.delaxes(axs[j])

    # --- Remove individual legends (we will use a separate legend-only figure)
    # (No action needed because we did not call ax.legend inside plot_run)

    # Slightly tighter layout (no suptitle)
    plt.tight_layout()

    # Save group figure in pgf + pdf + png (preview)
    outbase = PLOT_DIR / f"{group_label.lower()}_group"
    fig.savefig(str(outbase) + ".pgf", bbox_inches="tight")
    fig.savefig(str(outbase) + ".pdf", bbox_inches="tight")
    #fig.savefig(outbase + ".png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    # Save a legend-only file once (you can reuse for all groups)
    save_legend_only(PLOT_DIR / "method_legend.pgf")
    save_legend_only(PLOT_DIR / "method_legend.pdf")

## Generate manuscript convergence figures

Generates grouped convergence plots for the Hexagonal, Triangular Ga-face, and Triangular N-face benchmarks. Add the final manuscript figure numbers here once fixed.


In [ ]:
plot_group("Hex",  hex_configs)
plot_group("TriG", triG_configs)
plot_group("TriN", triN_configs)
